# Size the pipeline to this machine, on synthetic cells

Open this notebook from its GitHub link on a Colab L4 runtime, the experiment's one GPU, at the
latest runtime version (Runtime > Change runtime type): <https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/notebooks/03_tune_hardware.ipynb>

The aim is a run that uses the machine usefully, not the fastest run. Each section runs one hot
path of the pipeline, measures how busy the GPU, the CPUs and the disk are meanwhile, names what
holds it back, and finds the smallest setting that lifts that limit. The verdict also says how long
the full run's training and searches take on this machine.

Nothing here touches the real data or Drive: the cells are synthetic, of the real shape (6 bands,
512 x 512 pixels), and the models have random weights. Times per decoder call and per update hold
for this hardware; how many calls a search makes differs for a trained model, so the projected
hours are estimates. Every candidate is checked against a reference to the bit, and only one that
keeps every bit counts as safe; one that moves a number is reported for a decision.

`requirements.txt` pins the versions the latest Colab runtime publishes. An older runtime version
ships another torch, 2.11.0 on 2026.10, so the setup stops there before installing anything.

1. Set the form in section 1.
2. Run Runtime > Run all.
3. Read the verdict in section 11: `TUNING: PASS` or `TUNING: FAIL`, then the findings.

| Section | What it measures | Resource |
|---|---|---|
| 1. Form | What to measure | |
| 2. Setup | Checks Colab's versions against `requirements.txt`, clones the branch and installs it | disk |
| 3. Hardware | The L4, CPUs and RAM, the software stack and cuDNN flags in effect; the tests when asked | all |
| 4. Data loading | Cells per second from the disk at each worker count, read as the stages and as train read | CPU, disk |
| 5. Training | Time per update, data wait and GPU use of train's own loop at each worker count | GPU, CPU |
| 6. Search | `Memo` against the three separate searches on the same inputs: the bits, decoder calls, GPU use | GPU |
| 7. Batch size | GPU use, memory and time of each eval batch size, against the bits of batch 16 | GPU memory |
| 8. Precision | What strict float32 costs against the defaults and TF32, and which of them move the bits | GPU |
| 9. Determinism | Training and `compass_scores` under deterministic algorithms: an op without one, runs that differ | GPU |
| 10. LightGBM | calibrate's density ratio at each thread count | CPU |
| 11. Verdict | What limits each hot path, the setting that is enough, the projected hours, the regressions | |
| 12. Release | Releases the GPU, so the session stops spending compute units | |

Sections 3 to 10 measure in this notebook's own Python, the one the pipeline's commands run on:
the setup checks that the install keeps torch's CUDA build. Each runs
under the `%%section` cell magic of section 2, which records a failure and goes on, so the GPU is
released however a section ends. Their shared mechanics, timing and synthetic data, are in
`notebooks/tuning.py`, imported as `t`.

## 1. Form

- `BRANCH`: the code to measure.
- `UNET`, `CONVNET`: the architectures.
- `CELLS`: synthetic cells per measurement of sections 6 to 8, a multiple of every batch size.
- `SEARCH_UNET_LAYERS`: the U-Net layers of section 6; empty is every layer.
- `TIMED_UNET_LAYERS`: the U-Net layers of sections 7 and 8. `decoder.blocks.0` is the slowest
  layer of the pilot and `decoder.blocks.4` the one it chose.
- `LOADER_CELLS`: the cells of the synthetic store that sections 4 and 5 read from the disk.
- `WORKERS`: data-loader worker counts, or `auto` for 0, 2, 4, half and all of the CPUs.
- `BATCH_SIZES`, `STEPS`, `REPEATS`: the eval batch sizes of section 7, the updates per timed run
  of section 5, and the timed runs per candidate.
- `RUN_TESTS`: run the test suite on the GPU in section 3.
- `RELEASE`: release the GPU after the verdict.

In [ ]:
# @title The form
BRANCH = "main"  # @param {type:"string"}
UNET = True  # @param {type:"boolean"}
CONVNET = True  # @param {type:"boolean"}
CELLS = 64  # @param {type:"integer"}
SEARCH_UNET_LAYERS = ""  # @param {type:"string"}
TIMED_UNET_LAYERS = "decoder.blocks.0 decoder.blocks.4"  # @param {type:"string"}
LOADER_CELLS = 256  # @param {type:"integer"}
WORKERS = "auto"  # @param {type:"string"}
BATCH_SIZES = "16 32 64"  # @param {type:"string"}
STEPS = 20  # @param {type:"integer"}
REPEATS = 3  # @param {type:"integer"}
RUN_TESTS = False  # @param {type:"boolean"}
RELEASE = True  # @param {type:"boolean"}

## 2. Setup

Clones `BRANCH` into `/content/garimpo-uncertainty`, or moves the clone already there to the
branch's last commit, then installs `requirements.txt` into this Python, as the run notebook does.
Before the install it compares each package Colab ships with the version `requirements.txt` pins
for it, by pip's own rule. It stops when pip would replace torch or torchvision, whose CUDA build
picks the GPU's kernels. Any other difference, from a runtime a day or two behind its published
versions, pip aligns to the pins, and the setup names it, and also says which of those this Python
had already imported, which run their old version until a restart. It sets `CUBLAS_WORKSPACE_CONFIG` before
torch starts, which cuBLAS reads once, so that section 9 can run cuBLAS deterministically.
Outside Colab it uses the repository this notebook sits in, already installed. It also defines
`%%section`: a cell under it runs here, and a failure, or a failed setup, is noted for the verdict
instead of stopping Run all, so the GPU is still released at the end.

In [ ]:
# @title Setup
import importlib.metadata
import json
import os
import re
import site
import subprocess
import sys
import traceback
from pathlib import Path

from IPython.core.magic import register_cell_magic
from packaging.specifiers import SpecifierSet
from packaging.utils import canonicalize_name

# cuBLAS reads this once, at its first call; deterministic algorithms need it (section 9).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
ON_COLAB = "google.colab" in sys.modules
HERE = Path.cwd()
CODE = (
    Path("/content/garimpo-uncertainty")
    if ON_COLAB
    else next(path for path in [HERE, *HERE.parents] if (path / "pyproject.toml").exists())
)
OUT = CODE / "outputs" / "tuning"  # the synthetic store; erased with the session
FAILED = []  # every section or command that did not finish, with the reason
RESULTS = {}  # what each section measured, by section name, for the verdict


def shell(command: list[str], section: str, **options: object) -> int:
    """Run a command, streaming its output here; note a nonzero exit under `section`."""
    print("$", " ".join(command), flush=True)
    try:
        with subprocess.Popen(  # noqa: S603
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            encoding="utf-8",
            errors="replace",
            **options,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
        code = process.returncode
    except OSError as error:
        print(error)
        code = 127
    if code != 0:
        FAILED.append(f"{section}: {' '.join(command[:4])} ended with code {code}")
    return code


# The wheels that carry the CUDA libraries, which pick the GPU's kernels and so its bits.
CUDA_BUILDS = {"torch", "torchvision"}


def replaced_by_install(requirements: Path) -> list[tuple[str, str, str]]:
    """Return name, shipped version and pin of each Colab package the pins do not admit."""
    replaced = []
    for name, pin in re.findall(
        r"^([\w.-]+)==(\S+)$", requirements.read_text(encoding="utf-8"), re.MULTILINE
    ):
        try:
            shipped = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            continue  # not in Colab: pip installs it, replacing nothing
        if not SpecifierSet(f"=={pin}").contains(shipped, prereleases=True):
            replaced.append((canonicalize_name(name), shipped, pin))
    return replaced


def loaded_distributions() -> set[str]:
    """Name the distributions this Python has imported, which keep their version until a restart."""
    owners = importlib.metadata.packages_distributions()
    modules = {module.partition(".")[0] for module in list(sys.modules)}
    return {canonicalize_name(dist) for module in modules for dist in owners.get(module, [])}


def keeps_cuda_build(requirements: Path) -> bool:
    """Say what the install replaces; False, with the reason noted, if it swaps a CUDA build."""
    # Colab's runtime can trail its published versions by a day or two; pip then aligns those
    # packages to the pins, but another CUDA build would pick other kernels, so it stops there.
    named = {
        name: f"{name} {shipped} (pinned {pin})"
        for name, shipped, pin in replaced_by_install(requirements)
    }
    if blocking := [named[name] for name in named if name in CUDA_BUILDS]:
        FAILED.append(
            f"setup: pip would replace Colab's CUDA build, {', '.join(blocking)}: "
            "pick the latest runtime version"
        )
        print(FAILED[-1])
        return False
    if named:
        print("pip aligns to the pins:", ", ".join(named.values()))
    loaded_here = loaded_distributions()
    if loaded := [named[name] for name in named if name in loaded_here]:
        print("This Python keeps running the loaded", ", ".join(loaded), "until a restart")
    return True


def prepare() -> bool:
    """Bring the clone to BRANCH's last commit and install it into this Python; False on failure."""
    if not ON_COLAB:
        return True
    steps = [] if (CODE / ".git").is_dir() else [["git", "clone", REPO, str(CODE)]]
    steps += [
        ["git", "-C", str(CODE), "fetch", "origin", BRANCH],
        ["git", "-C", str(CODE), "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
        ["git", "-C", str(CODE), "submodule", "update", "--init", "--recursive"],
    ]
    if not all(shell(step, "setup") == 0 for step in steps):
        return False
    if not keeps_cuda_build(CODE / "requirements.txt"):
        return False
    # requirements.txt names ./compass and . from the clone's root.
    install = [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    return shell(install, "setup", cwd=CODE) == 0


def worker_counts() -> list[int]:
    if WORKERS.strip() != "auto":
        return sorted({0, *map(int, WORKERS.split())})
    cpus = len(os.sched_getaffinity(0))
    return sorted({0, 2, 4, cpus // 2, cpus} & set(range(cpus + 1)))


def form() -> dict:
    """Return the form as the sections read it, with the choices spelled out."""
    archs = [arch for arch, chosen in (("unet", UNET), ("convnet", CONVNET)) if chosen]
    workers = worker_counts()
    return {
        "seed": 0,
        "archs": archs,
        "cells": CELLS,
        "search_layers": {"unet": SEARCH_UNET_LAYERS.split(), "convnet": []},
        "timed_layers": {"unet": TIMED_UNET_LAYERS.split(), "convnet": []},
        "loader_cells": LOADER_CELLS,
        "workers": workers,
        "threads": sorted({1, *(count for count in workers if count > 0)}),
        "batch_sizes": [int(size) for size in BATCH_SIZES.split()],
        "steps": STEPS,
        "repeats": REPEATS,
    }


@register_cell_magic
def section(name: str, cell: str) -> None:
    """Run the cell here; a failure is noted under `name` and the later sections still run.

    A raise would stop Run all before section 12, and the GPU would stay assigned.
    """
    if not READY:
        print(f"Skipped {name}: the setup failed.")
        FAILED.append(f"{name}: skipped, the setup failed")
        return
    try:
        exec(cell, get_ipython().user_ns)  # noqa: S102
    except (Exception, SystemExit) as error:  # noqa: BLE001
        traceback.print_exc()
        FAILED.append(f"{name}: {type(error).__name__}: {error}")


READY = prepare()
if READY:
    # pip wrote the editable installs' .pth files after this Python started, so it reads them now.
    for packages in site.getsitepackages():
        site.addsitedir(packages)
    sys.path.insert(0, str(CODE / "notebooks"))
    shell(["git", "-C", str(CODE), "log", "-1", "--format=Measuring commit %h %cs %s"], "setup")
FORM = form()
print("Form as the sections read it:", json.dumps(FORM, indent=2), sep="\n")

## 3. Hardware

What the session got, which must be the L4, the software that picks its kernels, and the cuDNN
flags every later section starts from. Two runs share bits only when that stack matches too. The test suite runs
afterwards when the form asks for it, so a failing test shows before any timing.

In [ ]:
%%section hardware
import statistics

import numpy as np
import torch

import tuning as t

t.require_gpu()
gpu = torch.cuda.get_device_properties(0)
found = {
    "gpu": gpu.name,
    "capability": list(torch.cuda.get_device_capability(0)),
    "gpu_memory_gb": round(gpu.total_memory / 2**30, 1),
    "cpus": len(os.sched_getaffinity(0)),
    "ram_gb": round(t.ram_gb(), 1),
    "stack": t.stack(),
    "defaults": t.current_flags() | {"deterministic": torch.backends.cudnn.deterministic},
}
RESULTS["hardware"] = found
print(json.dumps(found, indent=2))

In [ ]:
# @title The tests on the GPU, when the form asks for them
if RUN_TESTS and READY:
    tests = [sys.executable, "-m", "pytest", "-q"]
    shell(tests, "tests", cwd=CODE, env={**os.environ, "GARIMPO_TEST_DEVICE": "cuda"})
else:
    print("Skipped: RUN_TESTS is not checked, or the setup failed.")

## 4. Data loading

Writes a synthetic store of `LOADER_CELLS` cells to the local disk, then reads it onto the GPU with
each worker count, twice over: in order at the eval batch size, as the search stages read, and
shuffled and flipped at the training batch size, as train reads. The store's pages are dropped from
RAM before every read, so each one comes from the disk. A checksum of every cell, weighted by row
and by column so that a flip changes it, must match the 0-worker read for the stages; for train it
shows whether the flips depend on the worker count.

In [ ]:
%%section loader
store = OUT / "store"
cells = t.synthetic_store(store, FORM["loader_cells"], FORM["seed"])
rows = []
for as_train in (False, True):
    runs = {
        workers: lambda w=workers, a=as_train: t.read_cells(store, cells, w, as_train=a)
        for workers in FORM["workers"]
    }
    timed = t.time_candidates(runs, FORM["repeats"])
    for workers, found in timed.items():
        row = {"reads_as": "train" if as_train else "stages", "workers": workers}
        row |= t.summarized(found) | {"bitwise": t.same(found, timed[0])}
        row["cells_per_second"] = len(cells) / row["seconds"]["median"]
        rows.append(row)
        print(
            f"{row['reads_as']:6} {workers:2} workers: {row['cells_per_second']:5.0f} cells/s, "
            f"CPU busy {row['cpu_busy']:.0%}, I/O wait {row['cpu_iowait']:.0%}, "
            f"same bytes as 0 workers {row['bitwise']}"
        )
RESULTS["loader"] = {"rows": rows}

## 5. Training

`STEPS` updates of train's own `fit_epoch` from the same seeded start, at each worker count, reading
the store of section 4 from the disk. `fit_epoch` reports how long it waited for data; above 10% of
its time, the loop starves. The final weights show whether the worker count changes the training.

In [ ]:
%%section train
store = OUT / "store"
cells = t.synthetic_store(store, FORM["loader_cells"], FORM["seed"])
for arch in FORM["archs"]:
    loops = {workers: [] for workers in FORM["workers"]}
    runs = {
        workers: lambda w=workers, a=arch, log=loops[workers]: t.training_run(
            a, store, cells, w, FORM["steps"], loops=log
        )
        for workers in FORM["workers"]
    }
    timed = t.time_candidates(runs, FORM["repeats"])
    rows = []
    for workers, found in timed.items():
        row = {"workers": workers} | t.summarized(found)
        row["weights_match_0_workers"] = t.same(found, timed[0])
        if "seconds" in row:
            timed_loops = loops[workers][1:]  # the first is the untimed run's
            loop_seconds = statistics.median(s for s, *_ in timed_loops)
            row["seconds_per_update"] = loop_seconds / FORM["steps"]
            row["data_wait_share"] = statistics.median(wait / s for s, wait, _ in timed_loops)
            # The loop's own usage: the build and the weights' copy leave the GPU idle.
            for key in ("cpu_busy", "cpu_iowait"):
                row[key] = statistics.fmean(usage[key] for *_, usage in timed_loops)
            busy = [u["gpu_busy"] for *_, u in timed_loops if not np.isnan(u["gpu_busy"])]
            row["gpu_busy"] = statistics.fmean(busy or [np.nan])
            print(
                f"{arch} {workers:2} workers: {1e3 * row['seconds_per_update']:.0f} ms/update, "
                f"data wait {row['data_wait_share']:.0%}, GPU busy {row['gpu_busy']:.0%}, "
                f"CPU busy {row['cpu_busy']:.0%}"
            )
        rows.append(row)
    RESULTS[f"train-{arch}"] = {"rows": rows}

## 6. Search

`compass_scores`' searches on each layer, with `Memo`, the code of this branch, and with
compass2d's three separate searches, the code before it. Both run on the very same spaces of each
batch, from one encoder pass and one Jacobian, so a GPU that sums in another order between two
passes cannot pass for a change made by `Memo`: every score must keep its bits. The time per decoder
call is the hardware's; the number of calls depends on where a trained model's searches stop.

In [ ]:
%%section search
size = t.CONFIG["EVAL_BATCH_SIZE"]
batches = FORM["cells"] // size
for arch in FORM["archs"]:
    rows = []
    for bench in t.benches(arch, FORM["search_layers"][arch], FORM["cells"], FORM["seed"]):
        seconds = {name: [0.0] * FORM["repeats"] for name in t.SEARCHERS}
        counters = {name: t.DecodeCounter() for name in t.SEARCHERS}
        usage, bitwise = [], True
        for first in range(0, batches * size, size):
            inputs = t.batch_inputs(bench, slice(first, first + size))
            runs = {
                name: lambda s=searcher, c=counters[name], i=inputs: t.searched(i, s, c)
                for name, searcher in t.SEARCHERS.items()
            }
            timed = t.time_candidates(runs, FORM["repeats"])
            bitwise &= t.same(timed["memo"], timed["separate"]) and timed["memo"].stable
            usage += timed["memo"].usage
            for name, found in timed.items():
                runs = zip(seconds[name], found.seconds, strict=True)
                seconds[name] = [total + run for total, run in runs]
        # A pass of every batch is timed REPEATS times after one untimed pass, all decoding alike.
        decodes = {name: c.calls / (FORM["repeats"] + 1) for name, c in counters.items()}
        median = {name: statistics.median(runs) for name, runs in seconds.items()}
        row = {
            "layer": bench.layer,
            "memo_bitwise": bitwise,
            "decodes_per_batch": {name: decodes[name] / batches for name in t.SEARCHERS},
            "ms_per_decode": {name: 1e3 * median[name] / decodes[name] for name in t.SEARCHERS},
            "seconds_per_cell": {name: median[name] / (batches * size) for name in t.SEARCHERS},
            # A batch shorter than nvidia-smi's first sample has no reading.
            "gpu_busy": statistics.fmean(
                [r["gpu_busy"] for r in usage if not np.isnan(r["gpu_busy"])] or [np.nan]
            ),
            "gpu_memory_gb": max(run["gpu_memory_gb"] for run in usage),
        }
        rows.append(row)
        calls = row["decodes_per_batch"]
        print(
            f"{arch} {bench.layer}: same bits {bitwise}, decodes per batch "
            f"{calls['separate']:.0f} -> {calls['memo']:.0f}, "
            f"{row['ms_per_decode']['memo']:.1f} ms per decode, GPU busy {row['gpu_busy']:.0%}"
        )
    RESULTS[f"search-{arch}"] = {"rows": rows}

## 7. Batch size

All of `compass_scores`, encoder, Jacobian and searches, over `CELLS` cells at each batch size of the
form and at 16, the eval batch size of `configs/base.json`. A larger batch is worth it only while
the GPU is not yet busy at 16 and the batch leaves memory free; cuDNN may pick other kernels for it,
which moves the last bits, so the verdict keeps those sizes apart as a decision.

In [ ]:
%%section batch
reference = t.CONFIG["EVAL_BATCH_SIZE"]
sizes = sorted({reference, *FORM["batch_sizes"]})
if uneven := [size for size in sizes if FORM["cells"] % size]:
    raise SystemExit(f"CELLS {FORM['cells']} is no multiple of the batch sizes {uneven}")
for arch in FORM["archs"]:
    rows = []
    for bench in t.benches(arch, FORM["timed_layers"][arch], FORM["cells"], FORM["seed"]):
        runs = {size: lambda s=size, b=bench: t.score_pass(b, s) for size in sizes}
        timed = t.time_candidates(runs, FORM["repeats"])
        for size, found in timed.items():
            row = {"layer": bench.layer, "batch_size": size} | t.summarized(found)
            rows.append(row | {"bitwise": t.same(found, timed[reference])})
            if "seconds" in row:
                print(
                    f"{arch} {bench.layer} batch {size:3}: {row['seconds']['median']:.1f} s, "
                    f"GPU busy {row['gpu_busy']:.0%}, {row['gpu_memory_gb']:.1f} GB, "
                    f"same bits as {reference} {rows[-1]['bitwise']}"
                )
            else:
                print(f"{arch} {bench.layer} batch {size:3}: {row['failed']}")
    RESULTS[f"batch-{arch}"] = {"reference": reference, "rows": rows}

## 8. Precision

All of `compass_scores` at batch 16 under four sets of flags. Strict float32, with TF32 off on
convolutions and matmuls, is the reference: the protocol takes no reduced precision. The rows
answer what strict float32 costs, whether cuDNN's autotuning keeps its bits, and whether the
defaults, which this torch ships with TF32 on for convolutions, already differ from it. Strict
float32 runs `REPEATS` + 1 times, so bits that differ between its own runs are the GPU's noise.

In [ ]:
%%section precision
strict = {"cudnn_tf32": False, "matmul_tf32": False, "benchmark": False}
candidates = {
    "strict_fp32": strict,
    "strict_fp32_autotuned": strict | {"benchmark": True},
    "defaults": t.current_flags(),
    "tf32": {"cudnn_tf32": True, "matmul_tf32": True, "benchmark": False},
}


def flagged_pass(bench: t.Bench, flags: dict) -> dict:
    with t.backend_flags(**flags):
        return t.score_pass(bench, t.CONFIG["EVAL_BATCH_SIZE"])


for arch in FORM["archs"]:
    rows = []
    for bench in t.benches(arch, FORM["timed_layers"][arch], FORM["cells"], FORM["seed"]):
        runs = {name: lambda f=f, b=bench: flagged_pass(b, f) for name, f in candidates.items()}
        timed = t.time_candidates(runs, FORM["repeats"])
        for name, found in timed.items():
            row = {"layer": bench.layer, "candidate": name} | candidates[name] | t.summarized(found)
            rows.append(row | {"bitwise_with_strict_fp32": t.same(found, timed["strict_fp32"])})
            if "failed" in row:
                print(f"{arch} {bench.layer} {name:22}: {row['failed']}")
                continue
            print(
                f"{arch} {bench.layer} {name:22}: {row['seconds']['median']:.1f} s, "
                f"same bits as strict float32 {rows[-1]['bitwise_with_strict_fp32']}"
            )
    RESULTS[f"precision-{arch}"] = {"rows": rows}

## 9. Determinism

The protocol asks that one seed give one result on the L4. Here train's own `fit_epoch` and all
of `compass_scores` at batch 16 run under torch's deterministic algorithms, with the flags the
pipeline runs with today, the defaults of section 3, so this switch is the only change from
sections 5 and 7. An op that has no deterministic implementation on CUDA raises and is named;
otherwise every one of `REPEATS` + 1 runs must give the same bits. The training also runs at 0
and at 2 workers, whose weights differ only when the flips of section 4 depend on the worker
count, which is a finding. The time per update shows what determinism costs.

In [ ]:
%%section determinism
store = OUT / "store"
cells = t.synthetic_store(store, FORM["loader_cells"], FORM["seed"])
pair = sorted({0, max(count for count in FORM["workers"] if count <= 2)})


def under_determinism(candidates: dict) -> dict:
    """Time the candidates under deterministic algorithms, or name the op that has none."""
    try:
        with t.deterministic():
            return {"timed": t.time_candidates(candidates, FORM["repeats"])}
    except RuntimeError as error:
        if "deterministic" not in str(error):
            raise
        # Keep the op and drop torch's advice on turning the check off.
        return {"missing": str(error).splitlines()[0].split(", but you set")[0]}


for arch in FORM["archs"]:
    loops = {count: [] for count in pair}
    trainings = {
        count: lambda c=count, a=arch, log=loops[count]: t.training_run(
            a, store, cells, c, FORM["steps"], loops=log
        )
        for count in pair
    }
    checks = {"training": under_determinism(trainings)}
    for bench in t.benches(arch, FORM["timed_layers"][arch], FORM["cells"], FORM["seed"]):
        score = {bench.layer: lambda b=bench: t.score_pass(b, t.CONFIG["EVAL_BATCH_SIZE"])}
        checks[bench.layer] = under_determinism(score)
    rows = []
    for check, result in checks.items():
        row = {"check": check}
        if "missing" in result:
            row["missing"] = result["missing"]
            print(f"{arch} {check}: {row['missing']}")
            rows.append(row)
            continue
        timed = result["timed"]
        row["runs_agree"] = all(found.stable for found in timed.values())
        if check == "training":
            row["workers_agree"] = t.same(timed[pair[0]], timed[pair[-1]])
            timed_loops = loops[pair[-1]][1:]  # the first is the untimed run's
            row["workers"] = pair[-1]
            row["seconds_per_update"] = statistics.median(s for s, *_ in timed_loops) / FORM[
                "steps"
            ]
        rows.append(row)
        print(
            f"{arch} {check}: runs give the same bits {row['runs_agree']}"
            + (
                f", {pair} workers the same weights {row['workers_agree']}, "
                f"{1e3 * row['seconds_per_update']:.0f} ms/update"
                if check == "training"
                else ""
            )
        )
    RESULTS[f"determinism-{arch}"] = {"workers": pair, "rows": rows}

## 10. LightGBM

`calibrate`'s density ratio, the one CPU-heavy part of that stage, at each thread count, on
synthetic summed features as many as the full run's calibration and test cells, 16 and 256 wide:
the widths of the U-Net's `decoder.blocks.4` and `decoder.blocks.0`. The pipeline passes it as many
threads as data-loader workers, so this shows whether that many help or crowd each other.

In [ ]:
%%section lightgbm
rng = np.random.default_rng(FORM["seed"])
rows = []
for channels in (16, 256):
    features = t.synthetic_features(rng, channels)
    runs = {jobs: lambda j=jobs, f=features: t.density_ratio(f, j) for jobs in FORM["threads"]}
    timed = t.time_candidates(runs, FORM["repeats"])
    for jobs, found in timed.items():
        row = {"channels": channels, "jobs": jobs} | t.summarized(found)
        rows.append(row | {"bitwise": t.same(found, timed[FORM["threads"][0]])})
        print(
            f"{channels:3} channels, {jobs:2} threads: {row['seconds']['median']:.2f} s, "
            f"CPU busy {row['cpu_busy']:.0%}, same ratios as 1 thread {rows[-1]['bitwise']}"
        )
RESULTS["lightgbm"] = {"rows": rows}

## 11. Verdict

Reads every section's results. A failure is a missing section, a command that did not end with
code 0, a `Memo` score that lost a bit, a stage read that changed with the worker count, strict
float32 that gave other bits twice, or a training or search that the deterministic algorithms
cannot repeat to the bit: an op with no deterministic implementation, or runs that differ. The findings name what limits each hot path:

- data loading, when the loop waits on data more than 10% of its time;
- the GPU itself, when it is busy at least 80% of the time, which is the machine put to use;
- otherwise the CPU or kernel launches, which leave the GPU idle.

They also give the smallest setting that removes the limit, and the hours the full run's training
and searches take here. A candidate beats another only when their timed ranges do not overlap.

In [ ]:
# @title Verdict
import math

STARVED = 0.10  # a loop waiting on data longer than this share of its time starves
GPU_BUSY = 0.80  # below this busy share, the GPU waits on the CPU or on kernel launches
MEMORY_IN_USE = 0.80  # a batch leaves this share of GPU memory free; use varies by layer and cell


def read(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


# Without a clone there are no results, and nothing below reads these.
FULL = read(CODE / "configs" / "scales" / "full.json") if READY else {}
TILES = read(CODE / "outputs" / "split_summary.json")["tiles_by_scale_role"] if READY else {}
# 16 cells of 512 px per 2048 px tile; the search stages read the three evaluated roles.
EVALUATED_CELLS = 16 * sum(
    TILES.get(f"Artisanal/{role}", 0) for role in ("validation", "calibration", "test")
)
failures = list(FAILED)
findings = []


def limit(gpu_busy: float, data_wait: float = 0.0) -> str:
    if math.isnan(gpu_busy):
        return "GPU use not sampled: nvidia-smi gave no reading"
    if data_wait > STARVED:
        return f"limited by data loading, waiting {data_wait:.0%} of the time"
    if gpu_busy >= GPU_BUSY:
        return f"limited by the GPU, busy {gpu_busy:.0%}: the machine is put to use"
    return f"limited by the CPU or kernel launches: the GPU is busy only {gpu_busy:.0%}"


def overlap(first: dict, second: dict) -> bool:
    return first["low"] <= second["high"] and second["low"] <= first["high"]


def cheapest_tied(rows: list[dict], cost: str) -> dict | None:
    """Return the safe row of least `cost` whose timed range overlaps the fastest safe row's."""
    safe = [row for row in rows if "seconds" in row and row["bitwise"]]
    if not safe:
        return None
    fastest = min(safe, key=lambda row: row["seconds"]["median"])
    tied = [row for row in safe if overlap(row["seconds"], fastest["seconds"])]
    return min(tied, key=lambda row: row[cost])


def judge_loader(rows: list[dict]) -> None:
    stages = [row for row in rows if row["reads_as"] == "stages"]
    failures.extend(
        f"loader: {r['workers']} workers read other bytes" for r in stages if not r["bitwise"]
    )
    best = max(stages, key=lambda row: row["cells_per_second"])
    findings.append(
        f"loader: the stages read up to {best['cells_per_second']:.0f} cells/s from disk"
    )
    flipped = [r["workers"] for r in rows if r["reads_as"] == "train" and not r["bitwise"]]
    if flipped:
        findings.append(
            f"loader: train's flips differ at {flipped} workers, so NUM_WORKERS is part of a "
            "training's identity: keep it fixed across sessions and machines"
        )


def judge_train(arch: str, rows: list[dict]) -> None:
    rows = [row for row in rows if "seconds" in row]
    for row in rows:
        speed = f"{1e3 * row['seconds_per_update']:.0f} ms per update"
        findings.append(
            f"train {arch} {row['workers']} workers: {speed}, "
            + limit(row["gpu_busy"], row["data_wait_share"])
        )
    # Weights that differ between two runs at one worker count say nothing about the workers.
    if unstable := [row["workers"] for row in rows if not row["stable"]]:
        findings.append(
            f"train {arch}: two runs at {unstable} workers end with other weights, so the GPU's "
            "kernels vary under the defaults; section 9 runs it deterministically"
        )
    elif moved := [row["workers"] for row in rows if not row["weights_match_0_workers"]]:
        findings.append(f"train {arch}: {moved} workers end with other weights than 0 workers")
    fed = [row for row in rows if row["data_wait_share"] <= STARVED]
    if not fed:
        failures.append(f"train {arch}: data loading starves the loop at every worker count")
        return
    pick = min(fed, key=lambda row: row["workers"])
    updates = FULL["EPOCHS"] * FULL["STEPS_PER_EPOCH"]
    hours = pick["seconds_per_update"] * updates / 3600
    findings.append(
        f"train {arch}: {pick['workers']} workers keep it fed; {updates} updates take ~{hours:.1f} h"
    )


def judge_search(arch: str, rows: list[dict]) -> None:
    failures.extend(
        f"search {arch} {r['layer']}: Memo changed a score" for r in rows if not r["memo_bitwise"]
    )
    for row in rows:
        calls = row["decodes_per_batch"]
        findings.append(
            f"search {arch} {row['layer']}: {calls['separate']:.0f} -> {calls['memo']:.0f} decodes "
            f"per batch, {row['ms_per_decode']['memo']:.1f} ms each, " + limit(row["gpu_busy"])
        )
    hours = sum(row["seconds_per_cell"]["memo"] for row in rows) * EVALUATED_CELLS / 3600
    findings.append(
        f"search {arch}: these layers take ~{hours:.1f} h over the full run's {EVALUATED_CELLS} "
        "cells [estimate: a trained model's searches stop at other steps]"
    )


def judge_batch(arch: str, section: dict, memory: float) -> None:
    reference = section["reference"]
    for layer in dict.fromkeys(row["layer"] for row in section["rows"]):
        rows = [r for r in section["rows"] if r["layer"] == layer and "seconds" in r]
        base = next((r for r in rows if r["batch_size"] == reference), None)
        if base is None or not base["stable"]:
            failures.append(f"batch {arch} {layer}: batch {reference} failed or changed its bits")
            continue
        state = f"batch {arch} {layer}: {reference} keeps the GPU {base['gpu_busy']:.0%} busy in {base['gpu_memory_gb']:.1f} of {memory} GB"
        roomy = [r for r in rows if r["gpu_memory_gb"] <= MEMORY_IN_USE * memory]
        faster = [r for r in roomy if r["seconds"]["high"] < base["seconds"]["low"]]
        if base["gpu_busy"] >= GPU_BUSY or not faster:
            findings.append(f"{state}; no larger batch is worth its memory")
            continue
        pick = faster[0]
        moves = "" if pick["bitwise"] else ", but it moves numbers: a decision, not a setting"
        findings.append(f"{state}; batch {pick['batch_size']} is faster{moves}")


def judge_precision(arch: str, rows: list[dict]) -> None:
    for row in rows:
        if row["candidate"] == "strict_fp32" and not row["stable"]:
            failures.append(
                f"precision {arch} {row['layer']}: strict float32 gave other bits twice"
            )
        bits = (
            "the bits of strict float32"
            if row["bitwise_with_strict_fp32"]
            else "other bits than strict float32"
        )
        findings.append(
            f"precision {arch} {row['layer']} {row['candidate']}: {row['seconds']['median']:.1f} s, {bits}"
        )


def judge_determinism(arch: str, section: dict, trained: list[dict]) -> None:
    for row in section["rows"]:
        name = f"determinism {arch} {row['check']}"
        failures.extend(f"{name}: {problem}" for problem in determinism_problems(row))
        if row.get("workers_agree") is False:
            findings.append(
                f"{name}: {section['workers']} workers end with other weights, as the flips of "
                "section 4 depend on the worker count"
            )
        if "seconds_per_update" in row:
            findings.append(f"{name}: {determinism_cost(row, trained)}")


def determinism_problems(row: dict) -> list[str]:
    if "missing" in row:
        return [row["missing"]]
    return [] if row["runs_agree"] else ["two runs gave other bits under deterministic algorithms"]


def determinism_cost(row: dict, trained: list[dict]) -> str:
    cost = f"{1e3 * row['seconds_per_update']:.0f} ms per update at {row['workers']} workers"
    default = next((r for r in trained if r["workers"] == row["workers"] and "seconds" in r), None)
    if default is None:
        return cost
    return f"{cost}, against {1e3 * default['seconds_per_update']:.0f} ms under the defaults"


def judge_lightgbm(rows: list[dict]) -> None:
    for channels in dict.fromkeys(row["channels"] for row in rows):
        width = [row for row in rows if row["channels"] == channels]
        failures.extend(
            f"lightgbm {channels} channels: {r['jobs']} threads gave other ratios"
            for r in width
            if not r["bitwise"]
        )
        pick = cheapest_tied(width, "jobs")
        findings.append(f"lightgbm {channels} channels: {pick and pick['jobs']} threads are enough")


archs = FORM["archs"]
expected = ["hardware", "loader", "lightgbm"]
expected += [
    f"{kind}-{arch}"
    for kind in ("train", "search", "batch", "precision", "determinism")
    for arch in archs
]
failures.extend(
    f"{name}: no result, the section crashed or never ran"
    for name in expected
    if name not in RESULTS
)
if "hardware" in RESULTS:
    if "loader" in RESULTS:
        judge_loader(RESULTS["loader"]["rows"])
    if "lightgbm" in RESULTS:
        judge_lightgbm(RESULTS["lightgbm"]["rows"])
    for arch in archs:
        if f"train-{arch}" in RESULTS:
            judge_train(arch, RESULTS[f"train-{arch}"]["rows"])
        if f"search-{arch}" in RESULTS:
            judge_search(arch, RESULTS[f"search-{arch}"]["rows"])
        if f"batch-{arch}" in RESULTS:
            judge_batch(arch, RESULTS[f"batch-{arch}"], RESULTS["hardware"]["gpu_memory_gb"])
        if f"precision-{arch}" in RESULTS:
            judge_precision(
                arch, [r for r in RESULTS[f"precision-{arch}"]["rows"] if "seconds" in r]
            )
        if f"determinism-{arch}" in RESULTS:
            trained = RESULTS.get(f"train-{arch}", {}).get("rows", [])
            judge_determinism(arch, RESULTS[f"determinism-{arch}"], trained)
    hardware = RESULTS["hardware"]
    print(
        f"{hardware['gpu']}, {hardware['gpu_memory_gb']} GB; {hardware['cpus']} CPUs; {hardware['ram_gb']} GB RAM"
    )
    print("Stack:", ", ".join(f"{key} {value}" for key, value in hardware["stack"].items()))
print("Findings:", *(f"- {line}" for line in findings), sep="\n")
print("Failures:", *(f"- {line}" for line in failures or ["none"]), sep="\n")
print("TUNING: FAIL" if failures else "TUNING: PASS: every section ran, and nothing lost a bit")

## 12. Release

Releases the GPU when `RELEASE` is checked, which ends the session and every result with it, so
copy the verdict above first.

In [ ]:
# @title Release the GPU
if RELEASE and ON_COLAB:
    from google.colab import runtime

    runtime.unassign()
else:
    print("Kept: RELEASE is not checked, or this is not Colab.")